# SparqlCE training script — Wikidata

Same recipe as DBpedia (paper main branch), adapted for the Wikidata dataset:
- loss      = InfoNCE(in-batch) + 0.3 · MSE(cos(a,p), jaccard(a,p))
- optimizer = AdamW, differential LRs (CodeBERT 1e-5, GNN/proj 1e-4)
- selection = best validation MRR (not mean cosine — collapse-biased)
- splits    = paper's fixed `splits_wikidata.json` if present, else seeded random

## What changed vs the DBpedia notebook
**Wikidata-specific:**
- Paths point to `wikidata-triplets-dataset.json` and `splits_wikidata.json`.
- Checkpoints saved under `checkpoints/wikidata/`.
- Final model named `wikidata_model_mrr_<score>.pt`.

**Efficiency (training dynamics unchanged):**
- `DataLoader` uses multiple workers + pinned memory + persistent workers — important for Wikidata's larger graph-building load.
- `zero_grad(set_to_none=True)` for small speedup.
- Optional mixed-precision training (`USE_AMP`). **Default OFF** — enable only after confirming a baseline fp32 epoch trains correctly.

## Known limitation to expect
Wikidata predicates are opaque P-numbers (`P31`, `P569`, …). CodeBERT has no semantic prior over these tokens, so absolute MRR/F1 numbers will be lower than DBpedia even with a correct training run. This is a vocabulary issue, not a training bug. See §6 (Limitations) of the paper.

## Before running
Verify the three paths in the CONFIG cell.

In [1]:
import os
import sys
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from tqdm import tqdm
from pathlib import Path

In [2]:
# ============================================================================
# 1. CONFIG — verify paths before running
# ============================================================================
DATA_PATH   = "../../datasets/wikidata-triplets-dataset.json"
SPLITS_PATH = "../../datasets/splits_wikidata.json"
CKPT_DIR    = "../../checkpoints/wikidata"

# ---- Hyperparameters (match paper / DBpedia recipe) ----
SEED         = 42
EPOCHS       = 20
BATCH_SIZE   = 32
TEMPERATURE  = 0.05
LAMBDA_REG   = 0.3
WEIGHT_DECAY = 1e-2
GRAD_CLIP    = 1.0

# ---- Differential LRs ----
USE_DIFFERENTIAL_LR = True
LR_TEXT_ENCODER     = 1e-5
LR_REST             = 1e-4
LR_SINGLE           = 1e-5

# ---- Efficiency knobs (do NOT change training dynamics) ----
NUM_WORKERS  = 4         # parallelise graph construction across CPU workers
PIN_MEMORY   = True      # faster CPU→GPU transfer
PERSIST_WORK = True      # keep workers alive across epochs

# Mixed precision: disabled by default. Enable only AFTER confirming
# a baseline fp32 epoch reaches similar train loss as DBpedia did.
USE_AMP      = False
AMP_DTYPE    = torch.bfloat16   # bf16 is safer than fp16 on Ampere+ GPUs

In [3]:
# ============================================================================
# 2. Setup (paths, sys.path, seeds)
# ============================================================================
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = str(Path.cwd().parent.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from sparqlce.config import Config           # noqa: E402
from sparqlce.graph_builder import SPARQLGraphBuilder   # noqa: E402
from sparqlce.encoder import HybridSPARQLEncoder        # noqa: E402
from sparqlce.dataset import (              # noqa: E402
    SemanticSPARQLDataset,
    semantic_collate_fn_inbatch,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")
if USE_AMP:
    print(f"✓ Mixed precision ENABLED ({AMP_DTYPE})")
else:
    print("  Mixed precision disabled (default)")

/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Device: cuda
  Mixed precision disabled (default)


/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /local/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /local/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_cluster/_version_cuda.so
  import torch_geometric.typing
/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disa

In [4]:
# ============================================================================
# 3. Data: dataset + fixed split (paper) with safe random fallback
# ============================================================================
graph_builder = SPARQLGraphBuilder()
dataset = SemanticSPARQLDataset(
    DATA_PATH,
    graph_builder=graph_builder,
    in_batch_negatives=True,
)
print(f"✓ Dataset loaded: {len(dataset)} triplets from {DATA_PATH}")

if os.path.exists(SPLITS_PATH):
    with open(SPLITS_PATH, "r") as f:
        splits = json.load(f)
    train_idx = splits.get("train_idx", splits.get("train"))
    val_idx   = splits.get("val_idx",   splits.get("valid_idx", splits.get("val")))
    test_idx  = splits.get("test_idx",  splits.get("test"))
    assert train_idx and val_idx and test_idx, f"Could not parse splits from {SPLITS_PATH}"

    train_dataset = Subset(dataset, train_idx)
    val_dataset   = Subset(dataset, val_idx)
    test_dataset  = Subset(dataset, test_idx)
    print(f"✓ Using FIXED split from {SPLITS_PATH}")
else:
    print(f"⚠  Fixed split file not found at {SPLITS_PATH}")
    print(f"⚠  Falling back to seeded random_split — numbers will NOT match paper Table 4.")
    total   = len(dataset)
    n_train = int(0.80 * total)
    n_val   = int(0.10 * total)
    n_test  = total - n_train - n_val
    gen = torch.Generator().manual_seed(SEED)
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [n_train, n_val, n_test], generator=gen
    )

print(f"  train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")

# DataLoader with parallel workers + pinned memory.
# persistent_workers=True avoids re-spawning the worker pool every epoch,
# which matters on Wikidata where graph construction dominates batch time.
_loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    collate_fn=semantic_collate_fn_inbatch,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY and torch.cuda.is_available(),
    persistent_workers=PERSIST_WORK and NUM_WORKERS > 0,
)
train_loader = DataLoader(train_dataset, shuffle=True,  **_loader_kwargs)
val_loader   = DataLoader(val_dataset,   shuffle=False, **_loader_kwargs)
test_loader  = DataLoader(test_dataset,  shuffle=False, **_loader_kwargs)
print(f"✓ DataLoaders: num_workers={NUM_WORKERS}, pin_memory={PIN_MEMORY}, "
      f"persistent_workers={PERSIST_WORK and NUM_WORKERS > 0}")

Loading dataset from ../../datasets/wikidata-triplets-dataset.json...
Loaded 10735 anchor groups.
✓ Dataset loaded: 10735 triplets from ../../datasets/wikidata-triplets-dataset.json
✓ Using FIXED split from ../../datasets/splits_wikidata.json
  train=8588  val=1073  test=1074
✓ DataLoaders: num_workers=4, pin_memory=True, persistent_workers=True


In [5]:
# ============================================================================
# 4. Model + optimizer + (optional) AMP scaler
# ============================================================================
model = HybridSPARQLEncoder().to(device)

if USE_DIFFERENTIAL_LR:
    text_kw = ("text", "txt", "codebert", "bert")
    text_params = [p for n, p in model.named_parameters()
                   if any(k in n.lower() for k in text_kw) and p.requires_grad]
    rest_params = [p for n, p in model.named_parameters()
                   if not any(k in n.lower() for k in text_kw) and p.requires_grad]
    n_text = sum(p.numel() for p in text_params)
    n_rest = sum(p.numel() for p in rest_params)
    print(f"✓ Differential LRs: text={n_text:,} @ {LR_TEXT_ENCODER}, "
          f"other={n_rest:,} @ {LR_REST}")
    if n_text == 0:
        print("⚠  No params matched text-encoder keywords; using single LR fallback.")
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR_SINGLE,
                                      weight_decay=WEIGHT_DECAY)
    else:
        optimizer = torch.optim.AdamW([
            {"params": text_params, "lr": LR_TEXT_ENCODER},
            {"params": rest_params, "lr": LR_REST},
        ], weight_decay=WEIGHT_DECAY)
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_SINGLE,
                                  weight_decay=WEIGHT_DECAY)

# AMP scaler: only needed for fp16. With bf16 it's effectively a no-op,
# but we keep it uniform so the training loop is identical either way.
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and AMP_DTYPE == torch.float16)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 927.16it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Differential LRs: text=109,580,544 @ 1e-05, other=6,402,944 @ 0.0001


/tmp/ipykernel_1444701/2434797438.py:31: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and AMP_DTYPE == torch.float16)


In [6]:
# ============================================================================
# 5. Loss: in-batch InfoNCE
# ============================================================================
def infonce_inbatch(a_emb, p_emb, temperature):
    a = F.normalize(a_emb, dim=1)
    p = F.normalize(p_emb, dim=1)
    logits = a @ p.T / temperature
    labels = torch.arange(a.size(0), device=a.device)
    return F.cross_entropy(logits, labels)

In [7]:
# ============================================================================
# 6. Validation: rank each anchor's true positive against ALL positives
# in the val split. Selects the embedding that retrieves correctly, not
# the one that collapses everything to a single point.
# ============================================================================
@torch.no_grad()
def evaluate_ranking(model, loader, device):
    model.eval()
    anchors, positives = [], []
    for batch in loader:
        a_emb, _ = model(batch[0].to(device))
        p_emb, _ = model(batch[1].to(device))
        anchors.append(a_emb.cpu())
        positives.append(p_emb.cpu())
    A = torch.cat(anchors)
    P = torch.cat(positives)
    sims  = A @ P.T
    diag  = sims.diag().unsqueeze(1)
    ranks = (sims >= diag).sum(dim=1).float()
    return {
        "mrr":    (1.0 / ranks).mean().item(),
        "hits@1": (ranks == 1).float().mean().item(),
        "hits@5": (ranks <= 5).float().mean().item(),
    }

In [8]:
# ============================================================================
# 7. Training loop
# ============================================================================
os.makedirs(CKPT_DIR, exist_ok=True)
best_mrr = -1.0

print(f"\n✓ Training on {device} for {EPOCHS} epochs")
print(f"  loss = InfoNCE(τ={TEMPERATURE}) + {LAMBDA_REG} · MSE(cos, jaccard)")

for epoch in range(1, EPOCHS + 1):
    model.train()
    sum_loss, sum_ce, sum_reg = 0.0, 0.0, 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS}")

    for batch in pbar:
        a_graphs = batch[0].to(device, non_blocking=PIN_MEMORY)
        p_graphs = batch[1].to(device, non_blocking=PIN_MEMORY)
        pos_js   = batch[2].to(device, non_blocking=PIN_MEMORY)

        optimizer.zero_grad(set_to_none=True)

        # Forward + loss under optional autocast
        with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
            a_emb, _ = model(a_graphs)
            p_emb, _ = model(p_graphs)
            
            loss_ce = infonce_inbatch(a_emb, p_emb, TEMPERATURE)

            a_n = F.normalize(a_emb, dim=1)
            p_n = F.normalize(p_emb, dim=1)
            cos = (a_n * p_n).sum(dim=1)
            loss_reg = (cos - pos_js).pow(2).mean()

            loss = loss_ce + LAMBDA_REG * loss_reg

        # Backward: scaler is a no-op when fp16 is not in use
        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            optimizer.step()

        sum_loss += loss.item()
        sum_ce   += loss_ce.item()
        sum_reg  += loss_reg.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         ce=f"{loss_ce.item():.4f}",
                         reg=f"{loss_reg.item():.4f}")

    nb_train = max(1, len(train_loader))
    val = evaluate_ranking(model, val_loader, device)
    print(f"Epoch {epoch:02d} | train_loss={sum_loss/nb_train:.4f} "
          f"(ce={sum_ce/nb_train:.4f} reg={sum_reg/nb_train:.4f}) "
          f"| val MRR={val['mrr']:.4f}  Hits@1={val['hits@1']:.4f}  Hits@5={val['hits@5']:.4f}")

    if val["mrr"] > best_mrr:
        best_mrr = val["mrr"]
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_metrics": val,
        }, os.path.join(CKPT_DIR, "best_model.pt"))
        print(f"  ✓ best saved (val MRR={best_mrr:.4f})")


✓ Training on cuda for 20 epochs
  loss = InfoNCE(τ=0.05) + 0.3 · MSE(cos, jaccard)


Epoch 01/20:   0%|          | 0/269 [00:00<?, ?it/s]/tmp/ipykernel_1444701/910559285.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
Epoch 01/20: 100%|██████████| 269/269 [03:18<00:00,  1.36it/s, ce=0.7828, loss=0.7862, reg=0.0114]


Epoch 01 | train_loss=1.5052 (ce=1.4983 reg=0.0231) | val MRR=0.3303  Hits@1=0.2675  Hits@5=0.3821
  ✓ best saved (val MRR=0.3303)


Epoch 02/20: 100%|██████████| 269/269 [03:17<00:00,  1.36it/s, ce=0.6835, loss=0.6862, reg=0.0089]


Epoch 02 | train_loss=1.2933 (ce=1.2873 reg=0.0199) | val MRR=0.3431  Hits@1=0.2740  Hits@5=0.3998
  ✓ best saved (val MRR=0.3431)


...

Epoch 18 | train_loss=0.6181 (ce=0.6099 reg=0.0275) | val MRR=0.4271  Hits@1=0.3299  Hits@5=0.5200
  ✓ best saved (val MRR=0.4271)


Epoch 19/20: 100%|██████████| 269/269 [03:17<00:00,  1.36it/s, ce=0.0890, loss=0.0988, reg=0.0328]


Epoch 19 | train_loss=0.6027 (ce=0.5942 reg=0.0282) | val MRR=0.4343  Hits@1=0.3420  Hits@5=0.5172
  ✓ best saved (val MRR=0.4343)


Epoch 20/20: 100%|██████████| 269/269 [03:17<00:00,  1.37it/s, ce=0.3151, loss=0.3218, reg=0.0223]


Epoch 20 | train_loss=0.5794 (ce=0.5706 reg=0.0291) | val MRR=0.4328  Hits@1=0.3327  Hits@5=0.5266


In [9]:
# ============================================================================
# 8. Final evaluation on the test split
# ============================================================================
print(f"\n✓ Training complete. Best val MRR: {best_mrr:.4f}")
print("✓ Reloading best checkpoint and evaluating on test split...")

CKPT_DIR    = "../../checkpoints/wikidata"
best = torch.load(os.path.join(CKPT_DIR, "best_model.pt"), map_location=device)
model.load_state_dict(best["model_state_dict"])
test = evaluate_ranking(model, test_loader, device)
print(f"TEST  MRR={test['mrr']:.4f}  Hits@1={test['hits@1']:.4f}  Hits@5={test['hits@5']:.4f}")


✓ Training complete. Best val MRR: 0.4343
✓ Reloading best checkpoint and evaluating on test split...
TEST  MRR=0.4407  Hits@1=0.3520  Hits@5=0.5279


In [10]:
# ============================================================================
# 9. Save the final model (named with dataset + test MRR for traceability)
# ============================================================================
save_dir = CKPT_DIR
os.makedirs(save_dir, exist_ok=True)

final_model_path = os.path.join(save_dir, f"wikidata_model_mrr_{test['mrr']:.4f}.pt")
torch.save(model.state_dict(), final_model_path)
print(f"✓ Final Wikidata model saved to: {final_model_path}")

✓ Final Wikidata model saved to: ../../checkpoints/wikidata/wikidata_model_mrr_0.4407.pt


In [ ]:
print(f"✓ Final Wikidata model saved to: {final_model_path}")